# GajahMada GNNM BumbleCore 300M — Google Colab GPU Training & Hugging Face Release (Release 2.6)

Notebook ini menjalankan alur **Release 2.6 Training Integrity** secara penuh di Google Colab GPU:
- **Model:** `GNNM-BumbleCore-300M` (`303,350,784` exact trainable parameters, BF16, Gradient Checkpointing)
- **Rekomendasi Runtime:** **Colab Pro GPU (L4 24GB / A100 40GB–80GB)**. Aktifkan via `Runtime → Change runtime type → GPU`.
- **Persistensi Checkpoint:** Disimpan ke **Google Drive** sehingga aman saat sesi Colab terputus (`--resume auto`).
- **Keamanan & Integritas:** Menggunakan korpus terbuka berlisensi (`CC-BY-SA-4.0`), `BoundedStreamingPreprocessor`, Checkpoint Trust Layer, uji paritas Safetensors, dan upload Hugging Face eksplisit (`--confirm-upload`).

In [ ]:
# 1. Cek GPU CUDA & Mount Google Drive untuk persistensi checkpoint
import importlib, os

torch = importlib.import_module('torch')
drive = importlib.import_module('google.colab').drive

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
assert torch.cuda.is_available(), 'Aktifkan GPU terlebih dahulu di Runtime -> Change runtime type -> GPU!'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/gajahmada_bumblecore_300m'
os.makedirs(os.path.join(DRIVE_ROOT, 'models'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, 'data'), exist_ok=True)
print('Google Drive workspace ready:', DRIVE_ROOT)

In [ ]:
# 2. Clone repository GajahMada AI & install dependensi
%cd /content
!if [ ! -d gajahmada-ai ]; then git clone https://github.com/hoed/gajahmada-ai.git; else cd gajahmada-ai && git pull origin main; fi
%cd /content/gajahmada-ai
%pip install -q -r requirements.txt datasets safetensors huggingface_hub

# Arahkan ./models langsung ke Google Drive agar checkpoint tersimpan permanen
!rm -rf /content/gajahmada-ai/models
!ln -s /content/drive/MyDrive/gajahmada_bumblecore_300m/models /content/gajahmada-ai/models
!ls -la /content/gajahmada-ai/models

In [ ]:
# 3. Unduh & kurasi korpus terbuka berlisensi (Wikipedia ID + EN, CC-BY-SA-4.0)
# Naikkan MAX_DOCS_PER_LANG untuk korpus penuh (mis. 500_000+ dokumen untuk pretraining skala penuh)
import importlib, json, os

load_dataset = importlib.import_module('datasets').load_dataset

DRIVE_DATA = '/content/drive/MyDrive/gajahmada_bumblecore_300m/data'
RAW_JSONL = os.path.join(DRIVE_DATA, 'raw_wiki_id_en.jsonl')
DATASET_DIR = os.path.join(DRIVE_DATA, 'pretrain_wiki_id_en_v1')
MAX_DOCS_PER_LANG = 50000  # Ubah sesuai target token & kapasitas Drive

if not os.path.isfile(os.path.join(DATASET_DIR, 'dataset_manifest.json')):
    written = 0
    with open(RAW_JSONL, 'w', encoding='utf-8') as out:
        for lang, cfg_name in (('id', '20231101.id'), ('en', '20231101.en')):
            ds = load_dataset('wikimedia/wikipedia', cfg_name, split='train', streaming=True)
            count = 0
            for row in ds:
                text = (row.get('text') or '').strip()
                if len(text) < 200:
                    continue
                rec = {
                    'text': text,
                    'source_doc_id': f'wiki-{lang}-{row.get("id", count)}',
                    'source_url': row.get('url'),
                    'language': lang,
                }
                out.write(json.dumps(rec, ensure_ascii=False) + '\n')
                count += 1
                written += 1
                if count >= MAX_DOCS_PER_LANG:
                    break
            print(f'Collected {count} {lang} documents.')
    print('Total raw JSONL documents written:', written)

    !python3 scripts/prepare_bumblecore_dataset.py \
      --input "{RAW_JSONL}" \
      --output-dir "{DATASET_DIR}" \
      --dataset-id bumblecore-wiki-id-en-v1 \
      --source-name wikimedia-wikipedia-20231101 \
      --source-url https://huggingface.co/datasets/wikimedia/wikipedia \
      --license CC-BY-SA-4.0 \
      --approval-status approved \
      --reviewer colab-operator
else:
    print('Verified dataset manifest already exists at:', DATASET_DIR)

In [ ]:
# 4. Resource Preflight Release 2.6 (memastikan VRAM & konfigurasi 303.35M valid)
!python3 -m gajahmada_ai.gnnm.train \
  --config configs/bumblecore_300m.yaml \
  --stage pretraining \
  --preflight

In [ ]:
# 5. Jalankan / Lanjutkan Pretraining GPU BumbleCore 300M (--resume auto ke Google Drive)
# Catatan: Hapus '--steps 200' jika ingin menjalankan full 50,000 steps (3.2768B token).
# Anda dapat menjalankan bertahap dan mengulang sel ini; --resume auto melanjutkan dari step terakhir.
DATASET_DIR = '/content/drive/MyDrive/gajahmada_bumblecore_300m/data/pretrain_wiki_id_en_v1'

!python3 -m gajahmada_ai.gnnm.train \
  --config configs/bumblecore_300m.yaml \
  --stage pretraining \
  --dataset "{DATASET_DIR}" \
  --resume auto \
  --save-every 5 \
  --steps 30

In [ ]:
# 6. Verifikasi Checkpoint Trust Layer, Export Safetensors, Paritas, & Dry-Run Hugging Face
# Ganti HF_REPO_ID dengan username/organisasi Hugging Face Anda
HF_REPO_ID = 'Hoedhud/GNNM_bumblecore_300m'

!python3 scripts/release_bumblecore_300m_hf.py \
  --checkpoint-dir ./models/bumblecore-300m/pretraining \
  --stage pretraining \
  --export-dir ./models/bumblecore-300m-export \
  --repo-id "{HF_REPO_ID}"

In [ ]:
# 7. Upload Resmi ke Hugging Face (Meminta HF_TOKEN secara aman — tidak tersimpan di notebook)
import getpass, importlib, os

HF_REPO_ID = 'Hoedhud/GNNM_bumblecore_300m'  # Repository Hugging Face target Anda
MAKE_PUBLIC = True  # Repo publik (ubah ke False jika ingin private)

try:
    token = importlib.import_module('google.colab').userdata.get('HF_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass.getpass('Masukkan Hugging Face Write Token (hf_...): ').strip()

os.environ['HF_TOKEN'] = token
pub_flag = '--public' if MAKE_PUBLIC else ''

!python3 scripts/release_bumblecore_300m_hf.py \
  --checkpoint-dir ./models/bumblecore-300m/pretraining \
  --stage pretraining \
  --export-dir ./models/bumblecore-300m-export \
  --repo-id "{HF_REPO_ID}" \
  --confirm-upload {pub_flag}